In [1]:
!pip install -q ultralytics sahi ensemble-boxes

import os, json, time, gc, warnings, pickle
from pathlib import Path

import numpy as np
import torch
import yaml
from PIL import Image

warnings.filterwarnings("ignore")

print("torch :", torch.__version__)
print("CUDA  :", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB")

N_GPU = torch.cuda.device_count()
DEVICE_LIST = list(range(N_GPU)) if N_GPU > 1 else 0
print("training device arg:", DEVICE_LIST)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 5.1 MB/s eta 0:00:00
torch : 2.10.0+cu128
CUDA  : True
  GPU 0: Tesla T4  15.6 GB
  GPU 1: Tesla T4  15.6 GB
training device arg: [0, 1]


In [2]:
# ---- phase flags: exactly one True per session, or all False to evaluate ----
RUN_TRAIN_RTDETR       = True
RUN_TRAIN_YOLO_CONTROL = False

# ---- matched training budget: identical for both arms ----------------------
BUDGET = dict(
    imgsz   = 1024,
    epochs  = 40,
    batch   = 8 if N_GPU > 1 else 4,   # RT-DETR attention is memory-hungry at 1024
    workers = 4,
)

# ---- matched inference settings for the headline comparison ----------------
EVAL = dict(
    device     = "cuda:0",
    imgsz      = 1024,
    conf_floor = 0.01,
    max_det    = 300,
    batch      = 8,
    tta        = False,   # see notes above: not equally supported, so off for both
    wbf_iou    = 0.55,
    wbf_skip   = 0.001,
    wbf_conf   = "avg",
    area_conf  = 0.25,
    limit      = 0,       # set to ~100 for a smoke test
)

OUT = Path("/kaggle/working/citylens_compare")
OUT.mkdir(parents=True, exist_ok=True)
print("BUDGET:", BUDGET)
print("EVAL  :", EVAL)

BUDGET: {'imgsz': 1024, 'epochs': 40, 'batch': 8, 'workers': 4}
EVAL  : {'device': 'cuda:0', 'imgsz': 1024, 'conf_floor': 0.01, 'max_det': 300, 'batch': 8, 'tta': False, 'wbf_iou': 0.55, 'wbf_skip': 0.001, 'wbf_conf': 'avg', 'area_conf': 0.25, 'limit': 0}


In [3]:
INPUT = Path("/kaggle/input")

roots = [p for p in INPUT.rglob("*") if p.is_dir()
         and (p/"train"/"images").is_dir() and (p/"valid"/"images").is_dir()]
if not roots:
    raise SystemExit("No dataset root with train/images + valid/images found.")
DATASET_ROOT = roots[0]
print("Dataset root:", DATASET_ROOT)

HAS_TEST = (DATASET_ROOT/"test"/"images").is_dir() and (DATASET_ROOT/"test"/"labels").is_dir()
for split in ["train", "valid"] + (["test"] if HAS_TEST else []):
    n = len(list((DATASET_ROOT/split/"images").iterdir()))
    print(f"  {split:<6} {n:>5} images")
if not HAS_TEST:
    print("!! No test split — everything falls back to val. Say so if you quote it.")

DATA_YAML = "/kaggle/working/citylens_data.yaml"
with open(DATA_YAML, "w") as f:
    yaml.safe_dump({"path": str(DATASET_ROOT), "train": "train/images",
                    "val": "valid/images", "test": "test/images",
                    "nc": 1, "names": {0: "pothole"}}, f)
print("\n" + open(DATA_YAML).read())

Dataset root: /kaggle/input/datasets/abhinavsinha08/pothole-master-citylens2/Pothole_master
  train   5278 images
  valid    901 images
  test     504 images

names:
  0: pothole
nc: 1
path: /kaggle/input/datasets/abhinavsinha08/pothole-master-citylens2/Pothole_master
test: test/images
train: train/images
val: valid/images



In [4]:
all_pt = sorted(p for p in INPUT.rglob("*.pt") if p.stat().st_size > 1e6)
print(f"{len(all_pt)} checkpoint(s) found:")
for p in all_pt:
    print(f"  {p.name:<38} {p.stat().st_size/1e6:6.1f} MB   {p.parent}")

def find(*needles):
    for p in all_pt:
        n = p.name.lower()
        if all(x in n for x in needles):
            return p
    return None

ARMS = {}
rt   = find("rtdetr")
ctrl = find("yolo", "control")
stg  = find("run3") or find("best_pothole")

if rt:   ARMS["rtdetr (40ep, matched)"]        = rt
if ctrl: ARMS["yolo_control (40ep, matched)"]  = ctrl
if stg:  ARMS["yolo_staged (150ep, 3-stage)"]  = stg

print("\nComparison arms:")
for k, v in ARMS.items():
    print(f"  {k:<34} {v.name}")

unassigned = [p for p in all_pt if p not in ARMS.values()]
if unassigned:
    print("\nUnassigned (ignored):", [p.name for p in unassigned])

1 checkpoint(s) found:
  best.pt                                  51.2 MB   /kaggle/input/datasets/abhinavsinha08/bestmodel

Comparison arms:

Unassigned (ignored): ['best.pt']


In [5]:
%%writefile /kaggle/working/metrics.py
from __future__ import annotations
import numpy as np

AREA_BUCKETS = {"small": (0, 32**2), "medium": (32**2, 96**2), "large": (96**2, np.inf)}


def iou_matrix(preds, gts):
    if len(preds) == 0 or len(gts) == 0:
        return np.zeros((len(preds), len(gts)), dtype=np.float32)
    p, g = preds[:, None, :], gts[None, :, :]
    x1 = np.maximum(p[..., 0], g[..., 0]); y1 = np.maximum(p[..., 1], g[..., 1])
    x2 = np.minimum(p[..., 2], g[..., 2]); y2 = np.minimum(p[..., 3], g[..., 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    ap = (preds[:, 2] - preds[:, 0]) * (preds[:, 3] - preds[:, 1])
    ag = (gts[:, 2] - gts[:, 0]) * (gts[:, 3] - gts[:, 1])
    union = ap[:, None] + ag[None, :] - inter
    return np.where(union > 0, inter / np.maximum(union, 1e-9), 0.0)


def match_image(preds, scores, gts, iou_thr):
    """Greedy match, predictions taken in descending-confidence order."""
    order = np.argsort(-scores)
    preds, scores = preds[order], scores[order]
    ious = iou_matrix(preds, gts)
    tp = np.zeros(len(preds), dtype=bool)
    matched = np.full(len(preds), -1, dtype=int)
    taken = set()
    for i in range(len(preds)):
        best_j, best_iou = -1, iou_thr
        for j in range(len(gts)):
            if j in taken:
                continue
            if ious[i, j] >= best_iou:
                best_iou, best_j = ious[i, j], j
        if best_j >= 0:
            tp[i] = True; matched[i] = best_j; taken.add(best_j)
    return tp, matched, scores


def found_set(preds, scores, gts, iou_thr=0.5, conf=0.25):
    """Indices of ground-truth boxes this detector found at the given threshold."""
    preds = np.asarray(preds, dtype=np.float64).reshape(-1, 4)
    scores = np.asarray(scores, dtype=np.float64).reshape(-1)
    gts = np.asarray(gts, dtype=np.float64).reshape(-1, 4)
    keep = scores >= conf
    tp, matched, _ = match_image(preds[keep], scores[keep], gts, iou_thr)
    return {int(m) for m, t in zip(matched, tp) if t}


def average_precision(tp, scores, n_gt):
    """101-point interpolated AP (COCO)."""
    if n_gt == 0:
        return float("nan")
    if len(tp) == 0:
        return 0.0
    order = np.argsort(-scores)
    tp = tp[order]
    tp_cum, fp_cum = np.cumsum(tp), np.cumsum(~tp)
    recall = tp_cum / n_gt
    precision = tp_cum / np.maximum(tp_cum + fp_cum, 1e-9)
    precision = np.maximum.accumulate(precision[::-1])[::-1]
    grid = np.linspace(0, 1, 101)
    idx = np.searchsorted(recall, grid, side="left")
    return float(np.where(idx < len(precision),
                          precision[np.minimum(idx, len(precision) - 1)], 0.0).mean())


def _clean(dataset):
    return [{"preds":  np.asarray(d["preds"],  dtype=np.float64).reshape(-1, 4),
             "scores": np.asarray(d["scores"], dtype=np.float64).reshape(-1),
             "gts":    np.asarray(d["gts"],    dtype=np.float64).reshape(-1, 4)}
            for d in dataset]


def confidence_sweep(dataset, iou_thr=0.5, thresholds=None):
    if thresholds is None:
        thresholds = np.round(np.arange(0.05, 0.96, 0.05), 2)
    n_gt = sum(len(d["gts"]) for d in dataset)
    cache = [match_image(d["preds"], d["scores"], d["gts"], iou_thr) for d in dataset]
    rows = []
    for t in thresholds:
        tp = fp = 0
        for flags, _, sc in cache:
            keep = sc >= t
            tp += int(flags[keep].sum()); fp += int((~flags[keep]).sum())
        p = tp / max(tp + fp, 1e-9); r = tp / max(n_gt, 1e-9)
        rows.append({"conf": float(t), "tp": tp, "fp": fp, "fn": n_gt - tp,
                     "precision": p, "recall": r,
                     "f1": 2 * p * r / max(p + r, 1e-9)})
    return rows


def recall_by_area(dataset, iou_thr=0.5, conf=0.25):
    hits = {k: 0 for k in AREA_BUCKETS}; totals = {k: 0 for k in AREA_BUCKETS}
    for d in dataset:
        found = found_set(d["preds"], d["scores"], d["gts"], iou_thr, conf)
        for j, gt in enumerate(d["gts"]):
            area = (gt[2] - gt[0]) * (gt[3] - gt[1])
            for name, (lo, hi) in AREA_BUCKETS.items():
                if lo <= area < hi:
                    totals[name] += 1
                    if j in found:
                        hits[name] += 1
                    break
    return {k: {"recall": hits[k] / totals[k] if totals[k] else float("nan"),
                "n": totals[k]} for k in AREA_BUCKETS}


def evaluate(dataset, iou_thrs=None, area_conf=0.25):
    if iou_thrs is None:
        iou_thrs = np.arange(0.5, 1.0, 0.05)
    dataset = _clean(dataset)
    n_gt = sum(len(d["gts"]) for d in dataset)
    aps = {}
    for thr in iou_thrs:
        all_tp, all_sc = [], []
        for d in dataset:
            tp, _, sc = match_image(d["preds"], d["scores"], d["gts"], thr)
            all_tp.append(tp); all_sc.append(sc)
        tp = np.concatenate(all_tp) if all_tp else np.array([], dtype=bool)
        sc = np.concatenate(all_sc) if all_sc else np.array([])
        aps[round(float(thr), 2)] = average_precision(tp, sc, n_gt)
    sweep = confidence_sweep(dataset, 0.5)
    best = max(sweep, key=lambda r: r["f1"])
    return {"AP50": aps[0.5], "AP50_95": float(np.nanmean(list(aps.values()))),
            "AP_by_iou": aps, "n_gt": n_gt, "best_f1": best, "sweep": sweep,
            "recall_by_area": recall_by_area(dataset, 0.5, area_conf)}


def yolo_txt_to_xyxy(path, img_w, img_h, with_conf=False):
    boxes, scores = [], []
    try:
        lines = open(path).read().strip().splitlines()
    except FileNotFoundError:
        return np.zeros((0, 4)), np.zeros(0)
    for line in lines:
        v = line.split()
        if len(v) < 5:
            continue
        xc, yc, w, h = (float(x) for x in v[1:5])
        boxes.append([(xc - w/2)*img_w, (yc - h/2)*img_h,
                      (xc + w/2)*img_w, (yc + h/2)*img_h])
        scores.append(float(v[5]) if (with_conf and len(v) >= 6) else 1.0)
    return np.array(boxes).reshape(-1, 4), np.array(scores)

Writing /kaggle/working/metrics.py


In [6]:
import sys
sys.path.insert(0, "/kaggle/working")
import importlib, metrics
importlib.reload(metrics)
from metrics import evaluate, yolo_txt_to_xyxy, found_set

_ds = [{"preds": [[0,0,10,10],[20,20,30,30]], "scores": [0.9,0.8],
        "gts":   [[0,0,10,10],[20,20,30,30]]}]
assert abs(evaluate(_ds)["AP50"] - 1.0) < 1e-6
assert evaluate([{"preds": np.zeros((0,4)), "scores": np.zeros(0),
                  "gts": [[0,0,10,10]]}])["AP50"] == 0.0
assert found_set([[0,0,10,10]], [0.9], [[0,0,10,10],[50,50,60,60]]) == {0}
print("metrics.py self-check passed")

metrics.py self-check passed


In [7]:
AUG = dict(
    mosaic   = 1.0,
    close_mosaic = 10,
    mixup    = 0.0,
    copy_paste = 0.0,
    hsv_h = 0.015, hsv_s = 0.7, hsv_v = 0.4,
    degrees = 0.0,
    translate = 0.1,
    scale = 0.5,
    fliplr = 0.5,
    erasing = 0.0,
)
print("shared augmentation (identical for both arms):")
print(json.dumps(AUG, indent=2))

shared augmentation (identical for both arms):
{
  "mosaic": 1.0,
  "close_mosaic": 10,
  "mixup": 0.0,
  "copy_paste": 0.0,
  "hsv_h": 0.015,
  "hsv_s": 0.7,
  "hsv_v": 0.4,
  "degrees": 0.0,
  "translate": 0.1,
  "scale": 0.5,
  "fliplr": 0.5,
  "erasing": 0.0
}


In [8]:
if RUN_TRAIN_RTDETR:
    from ultralytics import RTDETR

    model = RTDETR("rtdetr-l.pt")
    res = model.train(
        data=DATA_YAML,
        imgsz=BUDGET["imgsz"], epochs=BUDGET["epochs"],
        batch=BUDGET["batch"], workers=BUDGET["workers"],
        device=DEVICE_LIST,

        optimizer="AdamW",
        lr0=1e-4, lrf=0.01, cos_lr=True,
        warmup_epochs=5, weight_decay=1e-4,

        cache="disk", amp=True, patience=40, save_period=5,
        project="citylens_compare", name="rtdetr_1024",
        **AUG,
    )
    import shutil
    shutil.copy(f"{res.save_dir}/weights/best.pt",
                "/kaggle/working/citylens_rtdetr_best.pt")
    print("\nDownload citylens_rtdetr_best.pt and upload it as a Kaggle Dataset, "
          "then run session 2.")
else:
    print("RUN_TRAIN_RTDETR is False — skipping.")

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
New https://pypi.org/project/ultralytics/8.4.121 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.120 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=disk, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/citylens_data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5,

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution

       1/40      7.14G      1.632     0.4389      1.037         24       1024: 100% ━━━━━━━━━━━━ 660/660 1.4it/s 8:03
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 57/57 2.5it/s 23.3s
                   all        901       2572   0.000862     0.0906   0.000186   2.57e-05

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution

       2/40      7.32G      1.371     0.5466      0.778         11       1024: 100% ━━━━━━━━━━━━ 660/660 1.4it/s 7:44
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 57/57 2.7it/s 20.8s
                   all        901       2572     0.0052      0.264    0.00264   0.000507

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution

      39/40       7.2G     0.6544     0.6192     0.3398          8       1024: 100% ━━━━━━━━━━━━ 660/660 1.4it/s 7:39
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 57/57 2.7it/s 21.2s
                   all        901       2572      0.624      0.579      0.576       0.26

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution

      40/40       7.2G     0.6471     0.6101     0.3241          3       1024: 100% ━━━━━━━━━━━━ 660/660 1.4it/s 7:38
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 57/57 2.7it/s 20.9s
                   all        901       2572      0.627      0.581      0.575      0.261

40 epochs completed in 5.377 hours.
Optimizer stripped from /kaggle/working/runs/detect/citylens_compare/rtdetr_1024/weights/last.pt, 66.3MB
Optimizer stripped from /kaggle/working/runs/detect/citylens_compare/rtdetr_1024/weights/best.pt, 66.3MB

Validating /kaggle/working/runs/detect/citylens_compare/rtdetr_1024/weights/best.pt...
rt-detr-l summary: 315 layers, 31,985,795 parameters, 0 gradients, 105.3 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 57/57 3.1it/s 18.4s
                   all        901       2572      0.616      0.579      0.575      0.263
Speed: 0.3ms preprocess, 18.5ms i

AttributeError: 'dict' object has no attribute 'save_dir'

In [ ]:
if RUN_TRAIN_YOLO_CONTROL:
    from ultralytics import YOLO

    model = YOLO("yolo11l.pt")
    res = model.train(
        data=DATA_YAML,
        imgsz=BUDGET["imgsz"], epochs=BUDGET["epochs"],
        batch=BUDGET["batch"], workers=BUDGET["workers"],
        device=DEVICE_LIST,

        optimizer="AdamW",
        lr0=1e-3, lrf=0.01, cos_lr=True,
        warmup_epochs=5, weight_decay=5e-4,

        cache="disk", amp=True, patience=40, save_period=5,
        project="citylens_compare", name="yolo_control_1024",
        **AUG,
    )
    import shutil
    shutil.copy(f"{res.save_dir}/weights/best.pt",
                "/kaggle/working/citylens_yolo_control_best.pt")
    print("\nDownload citylens_yolo_control_best.pt and upload it as a Kaggle Dataset, "
          "then run session 3 with both flags False.")
else:
    print("RUN_TRAIN_YOLO_CONTROL is False — skipping.")

In [ ]:
if RUN_TRAIN_RTDETR or RUN_TRAIN_YOLO_CONTROL:
    raise SystemExit("A training flag is set — evaluation cells are for session 3. "
                     "Set both flags False and re-run.")

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def load_split(split):
    img_dir = DATASET_ROOT / split / "images"
    lbl_dir = DATASET_ROOT / split / "labels"
    images = sorted(p for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXTS)
    if EVAL["limit"]:
        images = images[:EVAL["limit"]]
    gts, sizes = [], []
    for p in images:
        w, h = Image.open(p).size
        sizes.append((w, h))
        b, _ = yolo_txt_to_xyxy(lbl_dir / f"{p.stem}.txt", w, h)
        gts.append(b)
    return dict(images=images, gts=gts, sizes=sizes)

SPLITS = {s: load_split(s) for s in (["valid"] + (["test"] if HAS_TEST else []))}
for s, d in SPLITS.items():
    print(f"{s:<6} {len(d['images']):>5} images  {sum(len(g) for g in d['gts']):>6} boxes")

In [ ]:
from ultralytics import YOLO, RTDETR

def load_model(path):
    """RT-DETR and YOLO checkpoints need different classes; pick by task metadata."""
    name = Path(path).name.lower()
    if "rtdetr" in name:
        return RTDETR(str(path))
    return YOLO(str(path))


def predict(path, images, imgsz=None, tta=None):
    imgsz = EVAL["imgsz"] if imgsz is None else imgsz
    tta   = EVAL["tta"]   if tta   is None else tta
    model = load_model(path)
    out, B = [], EVAL["batch"]
    for i in range(0, len(images), B):
        batch = [str(p) for p in images[i:i+B]]
        kw = dict(imgsz=imgsz, conf=EVAL["conf_floor"], max_det=EVAL["max_det"],
                  device=EVAL["device"], verbose=False)
        if tta:
            kw["augment"] = True
        for r in model.predict(batch, **kw):
            b = r.boxes
            if b is not None and len(b):
                out.append((b.xyxy.cpu().numpy().astype(np.float64),
                            b.conf.cpu().numpy().astype(np.float64)))
            else:
                out.append((np.zeros((0, 4)), np.zeros(0)))
    del model
    gc.collect(); torch.cuda.empty_cache()
    return out


PREDS = {s: {} for s in SPLITS}
for split, d in SPLITS.items():
    print(f"\n[{split}]")
    for arm, ckpt in ARMS.items():
        t0 = time.time()
        PREDS[split][arm] = predict(ckpt, d["images"])
        n = sum(len(p) for p, _ in PREDS[split][arm])
        print(f"  {arm:<34} {time.time()-t0:6.1f}s   {n:>7} boxes")

In [ ]:
def score(split, preds):
    d = SPLITS[split]
    return evaluate([{"preds": p, "scores": s, "gts": g}
                     for (p, s), g in zip(preds, d["gts"])],
                    area_conf=EVAL["area_conf"])

RESULTS = {s: {} for s in SPLITS}
for split in SPLITS:
    print(f"\n{'='*76}\n{split.upper()} — matched settings "
          f"({EVAL['imgsz']}px, TTA={EVAL['tta']})\n{'='*76}")
    for arm in ARMS:
        r = score(split, PREDS[split][arm])
        RESULTS[split][arm] = r
        b, a = r["best_f1"], r["recall_by_area"]
        print("{:<34} AP50 {:.4f}  AP50-95 {:.4f}  F1 {:.4f}  sm-R {:.3f}".format(
            arm, r["AP50"], r["AP50_95"], b["f1"], a["small"]["recall"]))

In [ ]:
def complementarity(split, arm_a, arm_b, conf=None):
    conf = EVAL["area_conf"] if conf is None else conf
    d = SPLITS[split]
    both = only_a = only_b = neither = 0
    for i, gts in enumerate(d["gts"]):
        if len(gts) == 0:
            continue
        fa = found_set(*PREDS[split][arm_a][i], gts, 0.5, conf)
        fb = found_set(*PREDS[split][arm_b][i], gts, 0.5, conf)
        for j in range(len(gts)):
            ia, ib = j in fa, j in fb
            both    += ia and ib
            only_a  += ia and not ib
            only_b  += ib and not ia
            neither += not ia and not ib
    total = both + only_a + only_b + neither
    return {
        "total_gt": total,
        "both": both, "only_a": only_a, "only_b": only_b, "neither": neither,
        "recall_a": (both + only_a) / max(total, 1),
        "recall_b": (both + only_b) / max(total, 1),
        "oracle_recall": (both + only_a + only_b) / max(total, 1),
        "disagreement": (only_a + only_b) / max(total, 1),
    }


A = "yolo_control (40ep, matched)"
B = "rtdetr (40ep, matched)"

if A in ARMS and B in ARMS:
    for split in SPLITS:
        c = complementarity(split, A, B)
        head = max(c["recall_a"], c["recall_b"])
        print(f"\n--- {split} — {A} vs {B} @ conf {EVAL['area_conf']} ---")
        print("  ground-truth boxes      {}".format(c["total_gt"]))
        print("  found by both           {:>6}  ({:.1%})".format(c["both"], c["both"]/c["total_gt"]))
        print("  only control            {:>6}  ({:.1%})".format(c["only_a"], c["only_a"]/c["total_gt"]))
        print("  only RT-DETR            {:>6}  ({:.1%})".format(c["only_b"], c["only_b"]/c["total_gt"]))
        print("  missed by both          {:>6}  ({:.1%})".format(c["neither"], c["neither"]/c["total_gt"]))
        print("  ----")
        print("  best single recall      {:.4f}".format(head))
        print("  oracle recall (ceiling) {:.4f}".format(c["oracle_recall"]))
        print("  headroom fusion can win {:+.4f}".format(c["oracle_recall"] - head))
        print("  disagreement rate       {:.1%}".format(c["disagreement"]))
        if c["oracle_recall"] - head < 0.03:
            print("\n  >> Low headroom. These models fail on the same potholes; "
                  "fusion is not worth 2x inference. Report that.")
        else:
            print("\n  >> Real headroom. Fusion has something to exploit.")
else:
    print("Need both matched arms present to run this. Train them first.")

In [ ]:
from ensemble_boxes import weighted_boxes_fusion

def wbf_fuse(pred_sets, sizes, weights=None):
    n_img = len(pred_sets[0])
    assert all(len(s) == n_img for s in pred_sets)
    fused = []
    for i in range(n_img):
        w, h = sizes[i]
        bl, sl, ll = [], [], []
        for src in pred_sets:
            b, s = src[i]
            if len(b) == 0:
                bl.append([]); sl.append([]); ll.append([]); continue
            nb = np.asarray(b, dtype=np.float64).copy()
            nb[:, [0, 2]] = np.sort(nb[:, [0, 2]], axis=1)
            nb[:, [1, 3]] = np.sort(nb[:, [1, 3]], axis=1)
            nb[:, [0, 2]] /= w; nb[:, [1, 3]] /= h
            bl.append(np.clip(nb, 0, 1).tolist())
            sl.append(np.asarray(s, dtype=np.float64).tolist())
            ll.append([0] * len(s))
        if all(len(x) == 0 for x in bl):
            fused.append((np.zeros((0, 4)), np.zeros(0))); continue
        fb, fs, _ = weighted_boxes_fusion(
            bl, sl, ll, weights=weights,
            iou_thr=EVAL["wbf_iou"], skip_box_thr=EVAL["wbf_skip"],
            conf_type=EVAL["wbf_conf"])
        fb = np.array(fb, dtype=np.float64).reshape(-1, 4)
        fb[:, [0, 2]] *= w; fb[:, [1, 3]] *= h
        fused.append((fb, np.array(fs, dtype=np.float64)))
    return fused


FUSION_NAME = "fusion (control + RT-DETR, WBF)"
if A in ARMS and B in ARMS:
    for split in SPLITS:
        f = wbf_fuse([PREDS[split][A], PREDS[split][B]], SPLITS[split]["sizes"])
        PREDS[split][FUSION_NAME] = f
        r = score(split, f)
        RESULTS[split][FUSION_NAME] = r
        best_single = max(RESULTS[split][x]["AP50"] for x in (A, B))
        print("[{}] fusion AP50 {:.4f}  AP50-95 {:.4f}   (best single AP50 {:.4f}, "
              "delta {:+.4f})".format(split, r["AP50"], r["AP50_95"],
                                      best_single, r["AP50"] - best_single))

In [ ]:
def operating_point(split, name, conf):
    return min(RESULTS[split][name]["sweep"], key=lambda x: abs(x["conf"] - conf))

FINAL = {}
for name in RESULTS["valid"]:
    conf = RESULTS["valid"][name]["best_f1"]["conf"]
    rep_split = "test" if HAS_TEST else "valid"
    r = RESULTS[rep_split][name]
    op = operating_point(rep_split, name, conf)
    FINAL[name] = {"chosen_conf": conf, "split": rep_split,
                   "AP50": r["AP50"], "AP50_95": r["AP50_95"],
                   "precision": op["precision"], "recall": op["recall"], "f1": op["f1"],
                   "recall_by_area": r["recall_by_area"]}

print(f"threshold selected on val, metrics reported on "
      f"{'TEST' if HAS_TEST else 'VAL (no test split!)'}\n")
for n, v in FINAL.items():
    print("{:<40} conf {:.2f}  AP50 {:.4f}  AP50-95 {:.4f}  P {:.3f}  R {:.3f}  F1 {:.3f}".format(
        n, v["chosen_conf"], v["AP50"], v["AP50_95"],
        v["precision"], v["recall"], v["f1"]))

In [ ]:
def measure_latency(path, sample, runs=40, warmup=10):
    model = load_model(path)
    for _ in range(warmup):
        model.predict(str(sample), imgsz=EVAL["imgsz"],
                      device=EVAL["device"], verbose=False)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(runs):
        model.predict(str(sample), imgsz=EVAL["imgsz"],
                      device=EVAL["device"], verbose=False)
    torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) / runs * 1000
    del model; gc.collect(); torch.cuda.empty_cache()
    return ms

sample = SPLITS["valid"]["images"][0]
LATENCY = {}
for arm, ck in ARMS.items():
    ms = measure_latency(ck, sample)
    LATENCY[arm] = {"ms": round(ms, 1), "fps": round(1000/ms, 1)}
    print("{:<40} {:7.1f} ms   {:5.1f} FPS".format(arm, ms, 1000/ms))

if FUSION_NAME in RESULTS["valid"]:
    tot = LATENCY.get(A, {}).get("ms", 0) + LATENCY.get(B, {}).get("ms", 0)
    LATENCY[FUSION_NAME] = {"ms": round(tot, 1), "fps": round(1000/max(tot, 1e-6), 1)}
    print("{:<40} {:7.1f} ms   {:5.1f} FPS  (sum of both passes)".format(
        FUSION_NAME, tot, 1000/max(tot, 1e-6)))

In [ ]:
def table():
    rows = ["| Model | Budget | AP50 | AP50-95 | P | R | F1 | small-R | ms |",
            "|---|---|---|---|---|---|---|---|---|"]
    budget = {
        "yolo_control (40ep, matched)": "40 ep @1024",
        "rtdetr (40ep, matched)":       "40 ep @1024",
        "yolo_staged (150ep, 3-stage)": "150 ep (3.75x)",
        FUSION_NAME:                    "two 40 ep models",
    }
    for n, v in FINAL.items():
        a = v["recall_by_area"]["small"]["recall"]
        ms = LATENCY.get(n, {}).get("ms", float("nan"))
        rows.append("| {} | {} | {:.4f} | {:.4f} | {:.3f} | {:.3f} | {:.3f} | {:.3f} | {:.0f} |".format(
            n, budget.get(n, "-"), v["AP50"], v["AP50_95"],
            v["precision"], v["recall"], v["f1"], a, ms))
    return "\n".join(rows)

t = table()
print(t)

payload = {
    "budget": BUDGET, "eval": {k: str(v) for k, v in EVAL.items()},
    "augmentation": AUG,
    "arms": {k: str(v) for k, v in ARMS.items()},
    "final": FINAL, "latency": LATENCY,
    "complementarity": (complementarity("test" if HAS_TEST else "valid", A, B)
                        if (A in ARMS and B in ARMS) else None),
    "results": {sp: {n: {k: v for k, v in r.items() if k != "sweep"}
                     for n, r in RESULTS[sp].items()} for sp in RESULTS},
}
(OUT / "comparison.json").write_text(json.dumps(payload, indent=2, default=float))
(OUT / "comparison_table.md").write_text(t)

import shutil
shutil.make_archive("/kaggle/working/citylens_comparison", "zip", OUT)
print("\nWrote citylens_comparison.zip")